# 02 Model training: all three loss options

Trains XGBoost and EBM for all three loss options in one loop:

| Option | Loss | Contribution scale |
|--------|------|--------------------|
| squared_error   | squared error    | rentals (exactly additive) |
| poisson_log     | Poisson deviance | log scale (exactly additive) |
| poisson_native  | Poisson deviance | rentals (approximate) |

Important methodological note: options 2 and 3 train the same model.
The difference only appears in notebook 03 during contribution extraction.

Hyperparameters are identical across all three options (for comparability).

## 1. Setup

In [ ]:
from __future__ import annotations

import sys, json, time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from xgboost import XGBRegressor
from interpret.glassbox import ExplainableBoostingRegressor

from utils import RANDOM_STATE, RESULTS_DIR
from utils.data import load_train_test
from utils.models import LOSS_OPTIONS, compute_metrics, save_model
from IPython.display import display

print(f"Loss options: {list(LOSS_OPTIONS)}")

## 2. Load data

In [ ]:
X_train, y_train, X_test, y_test = load_train_test()

print(f"X_train: {X_train.shape}  |  X_test: {X_test.shape}")
print(f"\ny_train distribution:")
print(y_train.describe()[["min","25%","50%","75%","max","mean"]].round(2).to_string())
print(f"\nCategorical features:  {[c for c in X_train.columns if X_train[c].dtype.name == 'category']}")
print(f"Numeric features:      {[c for c in X_train.columns if X_train[c].dtype.name != 'category']}")

## 3. Target distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(y_train, bins=60, edgecolor="white")
axes[0].set_title("cnt original scale")
axes[0].set_xlabel("Rentals per hour")
axes[1].hist(np.log1p(y_train), bins=60, edgecolor="white", color="C1")
axes[1].set_title("cnt log1p")
axes[1].set_xlabel("log(1 + rentals)")
for ax in axes:
    ax.set_ylabel("Frequency")
plt.tight_layout()
display(fig)
print(f"Skewness (original): {y_train.skew():.3f}")
print(f"Skewness (log1p):    {np.log1p(y_train).skew():.3f}")

## 4. Hyperparameters

Both models use identical parameters across all three loss options.

### XGBoost
* n_estimators=800, learning_rate=0.03: more trees with a smaller step size
  give better generalisation than 400 trees at 0.05
* max_depth=7
* n_jobs=-1

### EBM
* interactions=15: 5 more than before, EBM benefits a lot from pair interactions
* max_bins=512: finer binning boundaries, about 2x more granularity
* n_jobs=-1: EBM parallelises the feature rounds over cores

In [ ]:
xgb_params = dict(
    n_estimators=800,
    max_depth=7,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    reg_lambda=1.0,
    enable_categorical=True,
    tree_method="hist",
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbosity=0,
)

ebm_params = dict(
    interactions=15,
    max_bins=512,
    learning_rate=0.02,
    max_rounds=10000,
    early_stopping_rounds=100,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

## 5. Training loop

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
all_results = {}

for LOSS in LOSS_OPTIONS.values():
    print(f"\n{'='*60}")
    print(f"  {LOSS.label}")
    print(f"{'='*60}")

    # XGBoost
    xgb = XGBRegressor(objective=LOSS.xgb_objective, **xgb_params)
    t0 = time.time()
    xgb.fit(X_train, y_train)
    xgb_train_time = time.time() - t0
    print(f"  XGBoost trained in {xgb_train_time:.2f} s")

    # EBM
    ebm = ExplainableBoostingRegressor(objective=LOSS.ebm_objective, **ebm_params)
    t0 = time.time()
    ebm.fit(X_train, y_train)
    ebm_train_time = time.time() - t0
    print(f"  EBM     trained in {ebm_train_time:.2f} s")

    # Predictions + metrics
    pred_xgb = xgb.predict(X_test)
    pred_ebm = ebm.predict(X_test)
    metrics_xgb = compute_metrics(y_test, pred_xgb)
    metrics_ebm = compute_metrics(y_test, pred_ebm)

    print(f"  XGBoost  RMSE={metrics_xgb['rmse']:.2f}  R2={metrics_xgb['r2']:.4f}  "
          f"neg.preds={metrics_xgb['n_negative_predictions']}")
    print(f"  EBM      RMSE={metrics_ebm['rmse']:.2f}  R2={metrics_ebm['r2']:.4f}  "
          f"neg.preds={metrics_ebm['n_negative_predictions']}")

    # Save models
    save_model(xgb, "xgb", LOSS.key)
    save_model(ebm, "ebm", LOSS.key)

    # Save metrics JSON (read by 02b_Comparison.ipynb)
    result = {
        "loss_option": {
            "key":                LOSS.key,
            "label":              LOSS.label,
            "description":        LOSS.description,
            "ebm_objective":      LOSS.ebm_objective,
            "xgb_objective":      LOSS.xgb_objective,
            "contribution_space": LOSS.contribution_space,
        },
        "training_time_seconds": {
            "xgb": xgb_train_time,
            "ebm": ebm_train_time,
        },
        "metrics": {
            "xgb": metrics_xgb,
            "ebm": metrics_ebm,
        },
        "n_train": int(len(X_train)),
        "n_test":  int(len(X_test)),
        "hyperparameters": {
            "xgb": {**xgb_params, "objective": LOSS.xgb_objective},
            "ebm": {**ebm_params, "objective": LOSS.ebm_objective},
        },
    }
    out_path = RESULTS_DIR / f"model_metrics_{LOSS.key}.json"
    out_path.write_text(json.dumps(result, indent=2, default=str))
    all_results[LOSS.key] = result

print(f"\nDone. Saved: {list(LOSS_OPTIONS)}")

## 6. Results table

In [ ]:
rows = []
for key, r in all_results.items():
    for model_name in ("xgb", "ebm"):
        m = r["metrics"][model_name]
        rows.append({
            "Option":             r["loss_option"]["key"],
            "Model":              model_name.upper(),
            "RMSE":               round(m["rmse"], 2),
            "MAE":                round(m["mae"], 2),
            "R2":                 round(m["r2"], 4),
            "Poisson_dev":        round(m["poisson_deviance"], 4),
            "neg_preds":          m["n_negative_predictions"],
            "Training_s":         round(r["training_time_seconds"][model_name], 1),
        })

summary = pd.DataFrame(rows)
display(summary)

# Sanity check: options 2 and 3 must be identical at the model level
if "poisson_log" in all_results and "poisson_native" in all_results:
    for mn in ("xgb", "ebm"):
        m2 = all_results["poisson_log"]["metrics"][mn]
        m3 = all_results["poisson_native"]["metrics"][mn]
        diff = max(abs(m2[k] - m3[k]) for k in ("rmse", "mae", "r2"))
        status = "identical" if diff < 1e-9 else f"diff={diff:.2e}"
        print(f"Option 2 vs 3, {mn.upper()}: {status}")

## 7. Diagnostics: prediction vs truth

In [ ]:
fig, axes = plt.subplots(len(all_results), 2, figsize=(11, 4 * len(all_results)))
if len(all_results) == 1:
    axes = [axes]

for row_idx, (key, r) in enumerate(all_results.items()):
    from utils.models import load_models
    xgb_m, ebm_m = load_models(key)
    label = r["loss_option"]["key"]

    for col_idx, (model_obj, name) in enumerate([(xgb_m, "XGBoost"), (ebm_m, "EBM")]):
        ax = axes[row_idx][col_idx]
        pred = model_obj.predict(X_test)
        ax.scatter(y_test, pred, alpha=0.12, s=7)
        lim = max(float(y_test.max()), float(pred.max()))
        ax.plot([0, lim], [0, lim], "r--", lw=1)
        ax.axhline(0, color="gray", lw=0.5)
        ax.set_xlabel("y_true (cnt)")
        ax.set_ylabel("y_pred")
        m = compute_metrics(y_test, pred)
        ax.set_title(f"{label} | {name}  R2={m['r2']:.3f}")
        if m["n_negative_predictions"] > 0:
            ax.text(0.02, 0.95, f"{m['n_negative_predictions']} neg. preds",
                    transform=ax.transAxes, color="red", fontsize=9, va="top")

plt.tight_layout()
display(fig)